# တန်း 12 - Agent Scratchpad နှင့် စကားပြောသမိုင်း လျှော့ချခြင်း

ဒီနော့ဘုတ်က Microsoft Agent Framework ကိုအသုံးပြုပြီး စကားပြောရှည်လျားမှုတွေမှာ စာဝှက်မှုကိုဘယ်လိုစီမံနိုင်ကြောင်း ပြသပါတယ်။ စကားပြောတွေတိုးလာတာနဲ့တပြိုင်နက် တိုကင် အရေအတွက်က ပိုများလာပြီး မော်ဒယ်ရဲ့ စာဝှက် ကွန်မြူနတီကိုကျော်လွန်သွားနိုင်ပါတယ်။ ဒါကို **စာဝှက်အကျဉ်းချုပ် နည်းပညာ** နဲ့ **agent scratchpad** ဆိုတဲ့ အစိတ်အပိုင်းကို အသုံးပြုပြီး ပျမ်းမျှတောက်ပတဲ့ စာဝှက်စီမံချက်နဲ့ ဖြေရှင်းပါတယ်။

## သင်တွေ့ရှိမယ့် အချက်တွေ
1. **စာဝှက် စီမံခန့်ခွဲမှု ဆိုင်ရာ အရေးကြီးမှု** - တိုကင်ကန့်သတ်နဲ့ စာဝှက်ကွန်မြူနတီတို့ကို နားလည်ခြင်း
2. **စာဝှက်သိထားသည့် Agent များ** - မိမိတို့ စကားပြောစာဝှက်ကို ကိုယ်တိုင် စီမံနိုင်တဲ့ agent တွေဆောက်လုပ်ခြင်း
3. **စာဝှက်အကျဉ်းချုပ် နည်းပညာ** - စကားပြောသမိုင်းကို တစ်ခုချင်းစီ အကျဉ်းချုပ်ဖို့ ကိရိယာတွေ အသုံးပြုခြင်း
4. **Agent Scratchpad** - စာဝှက် လျှော့ချမှုအပြီးမှာလည်း တည်ခိုင်တံ့ခိုင်နေတဲ့ မှတ်ဉာဏ်

## လိုအပ်ချက်များ
- Azure OpenAI ကို အပြင်ပတ်ဝန်းကျင် အဆင့်များဖြင့် ဖွဲ့စည်းထားခြင်း
- ယခင်ဘာသာရပ်များမှ အခြေခံ agent အယူအဆများကို နားလည်ထားခြင်း


## စနစ်တပ်ဆင်ရန်


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity --quiet

In [ ]:
import os
import asyncio
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Load environment variables and create Azure AI Foundry provider
load_dotenv()

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ Azure AI Foundry provider configured")


## ဘာကြောင့် Context စီမံခန့်ခွဲမှုမှာ အရေးပါသလဲ

LLM တစ်ခုစီမှာ အကန့်အသတ်ရှိသော **context window** တစ်ခုရှိသည် — တစ်ခါတည်းတောင်းဆိုမှုအတွင်းတွင် ခွဲခြမ်းစိတ်ဖြာနိုင်သည့် token များ၏အများဆုံးအရေအတွက်။ မူလ အကြိမ်အတိုးများဖြင့် ဆက်လက်ပြောဆိုသည့်အခါ -

- **Token အရေအတွက်သည် အသုံးပြုသူ၏ မက်ဆေ့ခ်ျနဲ့ အစုံစမ်းသူ၏ ပြန်ဖြေချက် တစ်ခုချင်းစီအတွက် ကြိမ်ဖျောက်တိုးနေပုံဖြစ်သည်။**
- **Prompt token များသည် ကုန်ကျစရိတ်အမြင့်ဆုံးဖြစ်သည်** အကြောင်းမှာ စကားပြောသမိုင်းအနှစ်ချုပ်ကို အားလုံး ပြန်ပေးပို့ဖို့ ဖြစ်သည်။
- နောက်ဆုံးတွင် စကားပြောမှုသည် **context window အတက်ထိစွဲလျက်ရှိပြီး မော်ဒယ်သည် ပျောက်ကွယ်သို့မဟုတ် အမှားဖြစ်စေခြင်း ဖြစ်သည်။**

### Context ကို စီမံခန့်ခွဲရန် မဟာဗျူဟာများ

| မဟာဗျူဟာ | အသုံးပြုပုံ | စားနပ်ရိတ် |
|---|---|---|
| **စပ်ထားခြင်း** | အဟောင်းဆုံး မက်ဆေ့ခ်ျများကို လွှတ်ပစ်သည် | အစောပိုင်း context ပျောက်ပျက်သည် |
| **အကျဉ်းချုပ်ရေးခြင်း** | အဟောင်းမက်ဆေ့ခ်ျများကို အကျဉ်းဖွင့်ထုတ်ပြန်သည် | ခြုံငုံသတင်းသာရှိပြီး အချက်အလက်အချို့ ပျောက်ဆုံးနိုင်သည် |
| **Scratchpad / ဘေးပြင်မှတ်ဉာဏ်** | အရေးကြီး သတင်းအချက်အလက်များကို စကားပြောမှုအပြင်သိမ်းဆည်းသည် | ကိရိယာခေါ်ဆိုရန်လိုအပ်သော်လည်း လျော့နည်းမှုကို မခံနိုင်ကြသည် |

ဒီ notebook ထဲမှာ ကျွန်ုပ်တို့သည် **အကျဉ်းချုပ်ရေးခြင်း**နှင့် **scratchpad tool** ကို တွဲဖက်အသုံးပြုလိုက်သည်၊ ထို့ကြောင့် အိုင်ဂျင်သည် စကားပြောမှုသမိုင်းကို အကျဉ်းချိုင့်လိုသော်လည်း ဆက်လက်တည်ရှိနိုင်သည်။


## ပတ်ဝန်းကျင် အသိပညာ Agent တစ်ခု ဖန်တီးခြင်း


In [ ]:
agent = await provider.create_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("🤖 Context-aware travel planning agent created")

## ရေရှည် ဆက်သွယ်စကားပြောပုံကို သရုပ်ဆောင်ခြင်း

အကြောင်းအရာတွေ တက်ကြွစွာ ဆက်လက်ဖြစ်ထွန်းသွားမည့် မျိုးစုံသော နောက်တစ်ကြိမ် စကားပြောချက်ကို လိုက်ပါကြည့်ရအောင်။ ကိုယ်စားလှယ်သည် အဓိက အသေးစိတ် (စိတ်နှစ်သက်ချက်များ၊ ဘတ်ဂျက်၊ ခရီးသွားရက်များ) ကို စကားပြော တစ်ကြိမ်မပြတ် ထိန်းသိမ်းထားပြီး ဆက်လက်မှုကို ပြသနိုင်သင့်သည်။


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

၄င်းအေးဂျင့်သည် ယခင်အကြိမ်များမှ စကားဝိုင်းအကြောင်းအရာကို မေ့မလျော့ဘဲ သိထားသည်ကို သတိပြုပါ — ၎င်းသည် ဂျပန်နိုင်ငံ၊ ဆူရှီ၊ ဘုရားကျောင်းများ၊ ဓာတ်ပုံရိုက်ခြင်း၊ $၃၀၀၀ ဘတ်ဂျက်၊ တစ်ဦးတည်းခရီးသွားမှုနှင့် ဧပြီလကာလအကြောင်းကို အသိပညာရှိသည်။ စကားတစ်ခုတည်းတွင် ၎င်းသည် ကောင်းမွန်စွာ လုပ်ဆောင်နိုင်သည်၊ သို့သော် စကားဝိုင်းပိုများသွားသော်လည်း ပြီးခဲ့သည့်ရာဇဝင် အပြည့်အစုံကို ပြန်လည်ပို့ရခြင်းမှာ ဝန်ကြီးရှိပါသည်။

အကြောင်းအရာ စုပေါင်းမှုကို မျှော်လင့်နိုင်ရန် အပိုဆက်အပိုကြိမ်များဖြင့် စကားပြောဆိုမှုကို ဆက်လက်လုပ်ဆောင်ကြပါစို့။


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Context Summarization Pattern

စကားပြောဆိုမှု ကြီးထွားလာသည်နှင့်အမျှ၊ စုစုပေါင်းအကြောင်းအရာများကို တစ်ခုပြီးတစ်ခု ထုတ်လွှင့်စေသော **အကျဉ်းချုပ်ကိရိယာ** တစ်ခုကို အသုံးပြုနိုင်သည်။ အင်အားဖြစ်သူသည် ဤကိရိယာကို ခေါ်ယူကာ အရေးကြီးသော선호ကောင်းများကို မှတ်တမ်းတင်ရန် ယူဆောင်သည်၊ ထို့ကြောင့် အဟောင်းစာစောင်များ ပယ်ဖျက်သွားပါကပါ၊ အဓိက အချက်အလက်များသည် သိမ်းဆည်းထားသည်။

ဤပုံစံသည် ပိုမိုတိုးတက်သော သမိုင်းကြောင်းလျော့ပါးမှုအတွက် အခြေခံအဆောက်အအုံဖြစ်သည်။
1. အင်အားဖြစ်သူသည် စကားပြောဆိုမှုမှ အဓိကအချက်များကို သတ်မှတ်သည်
2. အကျဉ်းချုပ်ကိရိယာကို ခေါ်ယူကာ ထိုအချက်များကို သိမ်းဆည်းသည်
3. အဟောင်းစာစောင်များကို စိတ်ချလုံခြုံစွာ ဖယ်ရှားနိုင်သည်၊ အကျဉ်းချုပ်သည် အရေးကြီးသောအချက်များကို ဖမ်းဆီးသည်

အောက်တွင် အင်အားဖြစ်သူ သင်ယူထားသောအချက်များကို တိုကျပ်သောအကျဉ်းကို မှတ်တမ်းတင်ရန် ခေါ်ယူနိုင်သည့် `summarize_preferences` ကိရိယာကို သတ်မှတ်ထားသည်။


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = await provider.create_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("🤖 Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## အကျဉ်းချုပ်

ဤသင်ခန်းစာတွင် Microsoft Agent Framework ကိုအသုံးပြု၍ ရေရှည်ပြေးနေသော agent စကားပြောဆိုမှုများတွင် context ကိုဘယ်လိုစီမံခန့်ခွဲရမည်ကို သင်ယူခဲ့ပါသည်။

### အဓိကအစိတ်အပိုင်းများ
- **Context window များမှာ ကန့်သတ်ချက်ရှိသည်** — စကားပြောဆိုမှုမှတ်တမ်းအတွင်းရှိ တိုကင်တစ်ခုချင်းစီသည် ငွေကုန်ကျစေပြီး ကန့်သတ်ချက်ထဲ စုစည်းတွက်ချက်သွားသည်။
- **အကျဉ်းသုံးဆောင်မှုကိရိယာများ** သည် agent ကို စုပေါင်းထားသော context ကို တိုကင်အသုံးချမှုလျော့နည်းစေရန် အကျဉ်းချုပ်များအဖြစ် ဖွဲ့စည်းပေးကာ အချက်အလက်အရေးကြီးส่วนကို ထိန်းသိမ်းပေးသည်။
- **Agent scratchpads** သည် စကားပြောဆိုမှုကိုလျော့ချခြင်းမှ ဆင်းရဲခြင်းမရှိစေရန် အမြဲတမ်း ပြင်ပမှတ်ဉာဏ်ကို ပေးဆောင်သည်။

### သင်တည်ဆောက်ခဲ့သည်များ
- မျိုးစုံပြန်လည်ဆက်သွယ်မှုများတွင် ဆက်လက်တည်ရှိမှုကို ထိန်းသိမ်းသည့် **context-aware agent**  
- အသုံးပြုသူအချက်အလက်အရေးပါသော အချက်ကို တိုက်စပ်အကျဉ်းချုပ်ဖော်ပြသည့် **အကျဉ်းချုပ် စနစ်** (`summarize_preferences`)  
- context ထိန်းသိမ်းမှုနှင့် ပြောင်းလဲမှုကို ဖော်ပြသည့် **မျိုးစုံပြန်လည်ဆက်သွယ်မှု**

### အမှန်တကယ် အသုံးချနိုင်သည့် နေရာများ
- **ဖောက်သည်ဝန်ဆောင်မှု Bot များ**: ရေရှည်ထောက်ခံမှုအစည်းအဝေးများတွင် ကြိုက်နှစ်သက်မှုများကို မှတ်မိထားသည်  
- **ပုဂ္ဂိုလ်ရေးအကူအညီပေးသူများ**: context ကို ထပ်မံရှင်းပြစရာမလိုဘဲ လောလောဆယ်တိုးတက်နေသည့် ပရောဂျက်များကို နေရာချထားနိုင်သည်  
- **ပညာရေး ဆရာ/ဆရာမများ**: အတန်းသား ကြိုးပမ်းမှုများကို မျိုးစုံတုံ့ပြန်မှုများအတွင်း ထိန်းသိမ်းနိုင်သည်  

### နောက်တစ်ဆင့်များ
- ဖိုင်အခြေပြု ထိန်းသိမ်းမှုဖြင့် စုံလင်သော scratchpad ကိရိယာ တည်ဆောက်ရန်  
- အကျဉ်းချုပ်ပြီးနောက် အလိုအလျောက်သမိုင်းသတ်မှတ်မှု ထည့်သွင်းရန်  
- သို့တိက်သော မှတ်ဉာဏ်ရှာဖွေရေးအတွက် vector database များနှင့် ပေါင်းစပ်ရန်  
- စကားပြောဆိုမှုများကို ရက်အနည်းငယ်အကြာမှတဆင့် context နဲ့ ပြန်လည်စတင်နိုင်သော agent များ တည်ဆောက်ရန်


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**မသေချာချက်**  
ဤစာတမ်းကို AI ဘာသာပြန်ဝန်ဆောင်မှု [Co-op Translator](https://github.com/Azure/co-op-translator) ဖြင့် ဘာသာပြန်ထားပါသည်။ ကျွန်ုပ်တို့သည် တိကျမှုကိုကြိုးစားပြီးဖြစ်သော်လည်း အလိုအလျောက် ဘာသာပြန်ခြင်းတွင် အမှားများ သို့မဟုတ် တိကျမှုမရှိမှုများ ဖြစ်ပေါ်နိုင်ကြောင်း သတိပြုရန်လိုသည်။ မူလစာတမ်းကို မှန်ကန်သည့်အချက်အလက်ရင်းမြစ်အဖြစ် သတ်မှတ်စဉ်းစားသင့်ပါသည်။ အရေးကြီးသောသတင်းအချက်အလက်များအတွက် ပရော်ဖက်ရှင်နယ် လူသားဘာသာပြန်ခြင်းကို အကြံပြုပါသည်။ ဤဘာသာပြန်ခြင်းကို အသုံးပြုခြင်းကြောင့် ဖြစ်ပေါ်သည့် မှားနားမြင်သာမှုများနှင့် အနားယူခြင်းများအတွက် ကျွန်ုပ်တို့မှာ တာဝန်မယူပါ။
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
